In [2]:
import pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = '/content/drive/MyDrive/Project59'
R = pd.read_csv(f'{BASE}/results/per_signature_v4_ABCD.csv', low_memory=False)

print('rows :', f'{len(R):,}')
print('cols :', R.columns.tolist())

# is_home_tissue ko pakka bool banao (CSV se string aa sakta hai)
if R['is_home_tissue'].dtype == object:
    R['is_home_tissue'] = R['is_home_tissue'].astype(str).str.lower().isin(['true','1','yes'])

print()
print(R['category'].value_counts().reindex(list('ABCD')).to_string())
print('\nexpected: A 29 · B 717 · C 5,376 · D 33,874')

Mounted at /content/drive
rows : 39,996
cols : ['drug', 'lineage', 'tissue', 'is_home_tissue', 'n_M', 'n_F', 'n_signatures', 'NES', 'FDR', 'category', 'skip_reason']

category
A       29
B      717
C     5376
D    33874

expected: A 29 · B 717 · C 5,376 · D 33,874


In [3]:
import pandas as pd, numpy as np

S = pd.read_csv(f'{BASE}/results/signature_level_ABCD.csv', low_memory=False)
print('rows :', f'{len(S):,}', '  (expected 36,523)')
print('cols :', S.columns.tolist())

def pick(df, *names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(f'inme se koi nahi mila: {names}')

CAT  = pick(S, 'sig_cat', 'category', 'cat')
SEX  = pick(S, 'donor_sex', 'sex')
DRUG = pick(S, 'cmap_name', 'drug')
LIN  = pick(S, 'cell_lineage', 'lineage')
CELL = pick(S, 'cell_iname', 'cell_line')

print(f'\nusing -> cat={CAT}  sex={SEX}  drug={DRUG}  lineage={LIN}  cell={CELL}')
print('\nsex values :', S[SEX].value_counts(dropna=False).to_dict())

rows : 36,523   (expected 36,523)
cols : ['bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig', 'donor_sex', 'cell_type', 'cell_lineage', 'drug', 'lineage', 'sig_cat']

using -> cat=sig_cat  sex=donor_sex  drug=cmap_name  lineage=cell_lineage  cell=cell_iname

sex values : {'F': 22813, 'M': 13710}


In [4]:
order = ['A','B','C','D']

# ---- signatures ----
ct = pd.crosstab(S[CAT], S[SEX]).reindex(order).fillna(0).astype(int)
ct['total']  = ct.sum(axis=1)
ct['pct_F']  = (100 * ct.get('F', 0) / ct['total']).round(1)
ct['pct_M']  = (100 * ct.get('M', 0) / ct['total']).round(1)

print('='*64)
print('SIGNATURES  —  har category me kitne male / female')
print('='*64)
print(ct.to_string())
print(f'\ntotal : {ct["total"].sum():,}   (expected 36,523)')

# ---- cell lines (signatures nahi, alag donors) ----
cl = (S.groupby([CAT, SEX])[CELL].nunique()
        .unstack(fill_value=0).reindex(order).fillna(0).astype(int))
cl['total'] = cl.sum(axis=1)

print('\n' + '='*64)
print('CELL LINES  —  kitne alag-alag donors')
print('='*64)
print(cl.to_string())
print('\n(ek cell line kai categories me aa sakti hai, isliye total 135 se zyada ho sakta hai)')

# ---- Category A kahan se aaya ----
A = S[S[CAT] == 'A']
print('\n' + '='*64)
print('CATEGORY A  —  drug × lineage × sex')
print('='*64)
g = (A.groupby([DRUG, LIN, SEX]).size().unstack(fill_value=0))
g['total'] = g.sum(axis=1)
g['cell_lines'] = A.groupby([DRUG, LIN])[CELL].nunique()
print(g.sort_values('total', ascending=False).to_string())

SIGNATURES  —  har category me kitne male / female
donor_sex      F     M  total  pct_F  pct_M
sig_cat                                    
A           2426   230   2656   91.3    8.7
B           2958  2205   5163   57.3   42.7
C           1710  2974   4684   36.5   63.5
D          15719  8301  24020   65.4   34.6

total : 36,523   (expected 36,523)

CELL LINES  —  kitne alag-alag donors
donor_sex   F   M  total
sig_cat                 
A          17  18     35
B          59  52    111
C          60  53    113
D          76  57    133

(ek cell line kai categories me aa sakti hai, isliye total 135 se zyada ho sakta hai)

CATEGORY A  —  drug × lineage × sex
donor_sex                                           F    M  total  cell_lines
cmap_name    cell_lineage                                                    
bortezomib   breast                              1875    0   1875           7
             kidney                               450    5    455           2
             haematopoie

In [6]:
import os, pandas as pd, numpy as np

for f in sorted(os.listdir(f'{BASE}/results')):
    if f.endswith('.csv') and 'signature' in f.lower():
        mb = os.path.getsize(f'{BASE}/results/{f}') / 1e6
        print(f'   {f:<44} {mb:7.2f} MB')

   per_signature_categories.csv                    2.74 MB
   per_signature_v3_no_filter.csv                  3.23 MB
   per_signature_v4_ABCD.csv                       3.16 MB
   signature_ABCD_per_drug.csv                     0.00 MB
   signature_counts_per_drug.csv                   0.00 MB
   signature_level_ABCD.csv                       14.26 MB
   signature_share_per_drug.csv                    0.00 MB
   signatures_50drugs.csv                         13.42 MB
   signatures_per_drug_per_lineage.csv             0.02 MB


In [7]:
S = pd.read_csv(f'{BASE}/results/signature_level_ABCD.csv', low_memory=False)
print('rows :', f'{len(S):,}', '  (expected 36,523)')
print('cols :', S.columns.tolist())

def pick(df, *names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(f'inme se koi nahi mila: {names}  |  available: {df.columns.tolist()}')

CAT  = pick(S, 'sig_cat', 'category', 'cat')
SEX  = pick(S, 'donor_sex', 'sex')
DRUG = pick(S, 'cmap_name', 'drug')
LIN  = pick(S, 'cell_lineage', 'lineage')
CELL = pick(S, 'cell_iname', 'cell_line')

print(f'\nusing -> cat={CAT}  sex={SEX}  drug={DRUG}  lineage={LIN}  cell={CELL}')
print('sex values :', S[SEX].value_counts(dropna=False).to_dict())
print('cat values :', S[CAT].value_counts().to_dict())

rows : 36,523   (expected 36,523)
cols : ['bead_batch', 'nearest_dose', 'pert_dose', 'pert_dose_unit', 'pert_idose', 'pert_itime', 'pert_time', 'pert_time_unit', 'cell_mfc_name', 'pert_mfc_id', 'nsample', 'cc_q75', 'ss_ngene', 'tas', 'pct_self_rank_q25', 'wt', 'median_recall_rank_spearman', 'median_recall_rank_wtcs_50', 'median_recall_score_spearman', 'median_recall_score_wtcs_50', 'batch_effect_tstat', 'batch_effect_tstat_pct', 'is_hiq', 'qc_pass', 'pert_id', 'sig_id', 'pert_type', 'cell_iname', 'det_wells', 'det_plates', 'distil_ids', 'build_name', 'project_code', 'cmap_name', 'is_exemplar_sig', 'is_ncs_sig', 'is_null_sig', 'donor_sex', 'cell_type', 'cell_lineage', 'drug', 'lineage', 'sig_cat']

using -> cat=sig_cat  sex=donor_sex  drug=cmap_name  lineage=cell_lineage  cell=cell_iname
sex values : {'F': 22813, 'M': 13710}
cat values : {'D': 24020, 'B': 5163, 'C': 4684, 'A': 2656}


In [9]:
import pandas as pd, numpy as np

CAT, SEX, DRUG, LIN, CELL = 'sig_cat', 'donor_sex', 'cmap_name', 'cell_lineage', 'cell_iname'
order = ['A', 'B', 'C', 'D']
LABEL = {'A': 'A  home tissue, significant',
         'B': 'B  cross tissue, significant',
         'C': 'C  tested, not significant',
         'D': 'D  could not be tested'}

# ====================================================================
# 1.  SIGNATURES BY CATEGORY AND DONOR SEX
# ====================================================================
ct = pd.crosstab(S[CAT], S[SEX]).reindex(order).fillna(0).astype(int)
ct['total'] = ct.sum(axis=1)
ct['%_female'] = (100 * ct['F'] / ct['total']).round(1)
ct['%_male']   = (100 * ct['M'] / ct['total']).round(1)
ct.index = [LABEL[i] for i in ct.index]

print('='*78)
print('TABLE 1   Signatures by category and donor sex')
print('='*78)
print(ct.to_string())

tF = ct['F'].sum(); tM = ct['M'].sum(); tot = tF + tM
print(f'\nAll signatures      : {tot:,}   female {tF:,} ({100*tF/tot:.1f}%)   male {tM:,} ({100*tM/tot:.1f}%)')
print('Read each row against that baseline: a row above 62.5% female is')
print('female-enriched relative to the dataset, below it is male-enriched.')

# ====================================================================
# 2.  DISTINCT CELL LINES (DONORS), NOT SIGNATURES
# ====================================================================
cl = (S.groupby([CAT, SEX])[CELL].nunique()
        .unstack(fill_value=0).reindex(order).fillna(0).astype(int))
cl['total_lines'] = cl.sum(axis=1)
cl.index = [LABEL[i] for i in cl.index]

print('\n' + '='*78)
print('TABLE 2   Distinct cell lines (donors) contributing to each category')
print('='*78)
print(cl.to_string())
print('\nOne cell line yields many signatures because dose and timepoint vary,')
print('so Table 1 counts experiments while Table 2 counts people.')
print('A cell line can appear in more than one category, so columns may')
print(f'exceed the {S[CELL].nunique()} distinct cell lines in the study.')

# ====================================================================
# 3.  WHERE CATEGORY A COMES FROM
# ====================================================================
A = S[S[CAT] == 'A']
g = A.groupby([DRUG, LIN, SEX]).size().unstack(fill_value=0)
for c in ['F', 'M']:
    if c not in g.columns: g[c] = 0
g = g.rename(columns={'F': 'sigs_female', 'M': 'sigs_male'})
g['sigs_total'] = g['sigs_female'] + g['sigs_male']
g['lines_female'] = A[A[SEX] == 'F'].groupby([DRUG, LIN])[CELL].nunique()
g['lines_male']   = A[A[SEX] == 'M'].groupby([DRUG, LIN])[CELL].nunique()
g[['lines_female', 'lines_male']] = g[['lines_female', 'lines_male']].fillna(0).astype(int)
g['both_sexes_present'] = (g['lines_female'] > 0) & (g['lines_male'] > 0)

print('\n' + '='*78)
print(f'TABLE 3   Category A: {len(A):,} signatures, by drug and lineage')
print('='*78)
print(g.sort_values('sigs_total', ascending=False).to_string())

bal  = g[g['both_sexes_present']]['sigs_total'].sum()
nbal = len(A) - bal
print(f'\nFrom lineages with BOTH sexes present : {bal:>6,} signatures ({100*bal/len(A):.1f}%)')
print(f'From single-sex lineages              : {nbal:>6,} signatures ({100*nbal/len(A):.1f}%)')
print('\nA single-sex lineage cannot support a sex-difference claim: the')
print('enrichment score is computed correctly, but only one sex contributed')
print('drug data to it. These are reported separately, not removed.')

TABLE 1   Signatures by category and donor sex
donor_sex                         F     M  total  %_female  %_male
A  home tissue, significant    2426   230   2656      91.3     8.7
B  cross tissue, significant   2958  2205   5163      57.3    42.7
C  tested, not significant     1710  2974   4684      36.5    63.5
D  could not be tested        15719  8301  24020      65.4    34.6

All signatures      : 36,523   female 22,813 (62.5%)   male 13,710 (37.5%)
Read each row against that baseline: a row above 62.5% female is
female-enriched relative to the dataset, below it is male-enriched.

TABLE 2   Distinct cell lines (donors) contributing to each category
donor_sex                      F   M  total_lines
A  home tissue, significant   17  18           35
B  cross tissue, significant  59  52          111
C  tested, not significant    60  53          113
D  could not be tested        76  57          133

One cell line yields many signatures because dose and timepoint vary,
so Table 1 counts 

In [10]:
import pandas as pd, numpy as np

CAT, SEX, DRUG, LIN, CELL = 'sig_cat', 'donor_sex', 'cmap_name', 'cell_lineage', 'cell_iname'

# signatures per cell line, within each drug x lineage
pc = S.groupby([DRUG, LIN, CELL]).size().reset_index(name='n_sigs')

def eff_n(x):
    """Kish effective sample size: agar sab barabar to = n, agar ek hi line
       chhayi hui hai to -> 1. (sum n)^2 / sum(n^2)"""
    n = x.values.astype(float)
    return (n.sum() ** 2) / (n ** 2).sum()

pair = (pc.groupby([DRUG, LIN])['n_sigs']
          .agg(cell_lines='size', total_sigs='sum',
               biggest_line='max', smallest_line='min',
               effective_lines=eff_n)
          .reset_index())
pair['effective_lines'] = pair['effective_lines'].round(2)
pair['pct_from_biggest'] = (100*pair['biggest_line']/pair['total_sigs']).round(1)
pair['inflation'] = (pair['cell_lines']/pair['effective_lines']).round(2)

# sirf Category A wale pairs
Apairs = S[S[CAT]=='A'][[DRUG, LIN]].drop_duplicates()
out = pair.merge(Apairs, on=[DRUG, LIN])

print('='*92)
print('PSEUDOREPLICATION IN CATEGORY A PAIRS')
print('='*92)
print(out.sort_values('total_sigs', ascending=False).to_string(index=False))

print(f'\nCategory A pairs            : {len(out)}')
print(f'Real cell lines, summed     : {out["cell_lines"].sum()}')
print(f'Effective cell lines, summed: {out["effective_lines"].sum():.1f}')
print('\n"effective_lines" = how many independent donors the pair really behaves like.')
print('If it sits far below "cell_lines", a few lines are carrying the average.')

PSEUDOREPLICATION IN CATEGORY A PAIRS
   cmap_name                       cell_lineage  cell_lines  total_sigs  biggest_line  smallest_line  effective_lines  pct_from_biggest  inflation
  bortezomib                             breast           7        1875           813              6             2.89              43.4       2.42
  bortezomib                             kidney           2         455           450              5             1.02              98.9       1.96
  bortezomib haematopoietic_and_lymphoid_tissue          15         255            61              1             6.60              23.9       2.27
      YM-155                               skin           6          28            21              1             1.73              75.0       3.47
     emetine             central_nervous_system           3           9             3              3             3.00              33.3       1.00
  bortezomib             central_nervous_system           2           8         